In [ ]:
import matplotlib.pyplot
import pandas

In [ ]:
EXPERIMENT_NAME = "KEDA Prometheus"

def parse_replicas(file_path : str) -> list[int]:
    replicas : list[int] = []

    with (open(f"{file_path}/replicas.log")) as file:
        for line in file:
            parts : list[str] = line.split()
            if "webserver" in parts[0]:
                replicas.append(int(parts[6]))
    
    return replicas

def plot_replicas(file_path : str) -> None:
    replicas = parse_replicas(file_path)

    matplotlib.pyplot.plot(replicas, color="red", label=EXPERIMENT_NAME)
    matplotlib.pyplot.ylabel("Replica Count") 
    matplotlib.pyplot.legend()
    matplotlib.pyplot.tight_layout()
    matplotlib.pyplot.show()

def parse_p99(file_path : str) -> pandas.Series:
    columns : list[str] = ['timestamp_ns', 'status', 'latency_ns']
    data_frames : list[pandas.DataFrame] = [
        pandas.read_csv(f"{file_path}/vegeta1.log", usecols=[0, 1, 2], header=None, names=columns),
        pandas.read_csv(f"{file_path}/vegeta2.log", usecols=[0, 1, 2], header=None, names=columns),
        pandas.read_csv(f"{file_path}/vegeta3.log", usecols=[0, 1, 2], header=None, names=columns),
    ]
    combined_data_frame : pandas.DataFrame = pandas.concat(data_frames, ignore_index=True)

    combined_data_frame['timestamp_ns'] = pandas.to_datetime(combined_data_frame['timestamp_ns'], unit='ns')
    combined_data_frame = combined_data_frame.set_index('timestamp_ns')

    #return combined_data_frame['latency_ns'].resample('1s').quantile(0.99).expanding().mean() / 1e9
    return combined_data_frame['latency_ns'].resample('1s').quantile(0.99) / 1e9
    
def plot_p99(file_path : str) -> None:
    latencies = parse_p99(file_path)

    matplotlib.pyplot.plot(latencies.to_numpy(), color="red", label=EXPERIMENT_NAME)
    matplotlib.pyplot.ylabel("P99 Latency (s)")
    matplotlib.pyplot.legend()
    matplotlib.pyplot.tight_layout()
    matplotlib.pyplot.show()
    
def plot(file_path : str) -> None:
    replicas = parse_replicas(file_path)
    latencies = parse_p99(file_path)

    fig, ax1 = matplotlib.pyplot.subplots(figsize=(12, 5))

    ax1.set_xlabel("Time (samples)")
    ax1.set_ylabel("Replicas", color="tab:blue")
    ax1.plot(replicas, color="tab:blue", label="Replicas")
    ax1.tick_params(axis="y", labelcolor="tab:blue")
    ax1.set_xticks(range(0, max(len(replicas), len(latencies)), 60))

    ax2 = ax1.twinx()
    ax2.set_ylabel("Running Average P99 Latency (s)", color="tab:red")
    ax2.plot(latencies.to_numpy(), color="tab:red", label="P99 Latency")
    ax2.tick_params(axis="y", labelcolor="tab:red")

    matplotlib.pyplot.title("Replicas vs P99 (Burn = 30)")
    fig.tight_layout()
    fig.show()

def plot_comparison(experiments : list[tuple[str, str]]) -> None:
    fig, ax1 = matplotlib.pyplot.subplots(figsize=(12, 5))
    ax2 = ax1.twinx()

    colors = matplotlib.pyplot.rcParams['axes.prop_cycle'].by_key()['color']
    max_len = 0

    for i, (file_path, name) in enumerate(experiments):
        color = colors[i % len(colors)]
        replicas = parse_replicas(file_path)
        latencies = parse_p99(file_path)
        max_len = max(max_len, len(replicas), len(latencies))

        ax1.plot(replicas, color=color, linestyle="-", label=f"{name}")
        ax2.plot(latencies.to_numpy(), color=color, linestyle="--")

    ax1.set_xlabel("Time (samples)")
    ax1.set_ylabel("Replicas")
    ax1.set_xticks(range(0, max_len, 60))
    ax2.set_ylabel("P99 Latency (s)")

    lines1, labels1 = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper left")

    matplotlib.pyplot.title("Replicas vs P99 Comparison")
    fig.tight_layout()
    fig.show()

In [ ]:
run = 3

plot_comparison([
    (f"results/default_hpa_burn_20/experiment_run_{run}", "Default HPA"),
    (f"results/tuned_hpa_burn_20/experiment_run_{run}", "Tuned HPA"),
    (f"results/fixed_replica_count_burn_20/experiment_run_{run}", "Tuned HPA"),
])